In [7]:
import pandas as pd
from pathlib import Path

base = Path(r"C:\Users\tlsdn\OneDrive\Documents\바탕 화면\LLM")

# =========================
# 1. discharge.csv
# =========================
discharge = pd.read_csv(base / "discharge.csv")
discharge = discharge.dropna(subset=["hadm_id", "text"]).copy()
discharge["hadm_id"] = pd.to_numeric(discharge["hadm_id"], errors="coerce")
discharge = discharge.dropna(subset=["hadm_id"])
discharge["hadm_id"] = discharge["hadm_id"].astype(int)

# =========================
# 2. radiology.csv
# =========================
radiology = pd.read_csv(base / "radiology.csv")
radiology = radiology.dropna(subset=["hadm_id", "text"]).copy()
radiology["hadm_id"] = pd.to_numeric(radiology["hadm_id"], errors="coerce")
radiology = radiology.dropna(subset=["hadm_id"])
radiology["hadm_id"] = radiology["hadm_id"].astype(int)

# =========================
# 3. diagnosis.csv
# =========================
diag = pd.read_csv(base / "diagnosis.csv", usecols=["stay_id", "icd_code"])
diag = diag.dropna(subset=["stay_id", "icd_code"]).copy()
diag["stay_id"] = pd.to_numeric(diag["stay_id"], errors="coerce")
diag = diag.dropna(subset=["stay_id"])
diag["stay_id"] = diag["stay_id"].astype(int)

# =========================
# 4. edstays.csv
# =========================
ed = pd.read_csv(base / "edstays.csv", usecols=["hadm_id", "stay_id"])
ed = ed.dropna(subset=["hadm_id", "stay_id"]).copy()
ed["hadm_id"] = pd.to_numeric(ed["hadm_id"], errors="coerce")
ed["stay_id"] = pd.to_numeric(ed["stay_id"], errors="coerce")
ed = ed.dropna(subset=["hadm_id", "stay_id"])
ed["hadm_id"] = ed["hadm_id"].astype(int)
ed["stay_id"] = ed["stay_id"].astype(int)

# =========================
# 5. 라벨 hadm_id 만들기
# =========================
diag_ed = diag.merge(ed, on="stay_id", how="inner")

label_hadm = set(diag_ed["hadm_id"].unique())
discharge_hadm = set(discharge["hadm_id"].unique())
radiology_hadm = set(radiology["hadm_id"].unique())
combined_hadm = discharge_hadm | radiology_hadm

# =========================
# 6. 결과 출력
# =========================
print("label hadm_id 수:", len(label_hadm))
print()

print("discharge hadm_id 수:", len(discharge_hadm))
print("discharge와 label 겹치는 hadm_id 수:", len(discharge_hadm & label_hadm))
print()

print("radiology hadm_id 수:", len(radiology_hadm))
print("radiology와 label 겹치는 hadm_id 수:", len(radiology_hadm & label_hadm))
print()

print("discharge + radiology 합친 hadm_id 수:", len(combined_hadm))
print("combined와 label 겹치는 hadm_id 수:", len(combined_hadm & label_hadm))

label hadm_id 수: 201657

discharge hadm_id 수: 331793
discharge와 label 겹치는 hadm_id 수: 154709

radiology hadm_id 수: 309670
radiology와 label 겹치는 hadm_id 수: 157191

discharge + radiology 합친 hadm_id 수: 374285
combined와 label 겹치는 hadm_id 수: 177676


In [8]:
# ============================================
# BASELINE 2 (One-Cell)
# radiology.csv + discharge.csv + Longformer
# ============================================

import os
import json
import time
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

# =========================
# CONFIG
# =========================
CONFIG = {
    "mimic_dir": r"C:\Users\tlsdn\OneDrive\Documents\바탕 화면\LLM",
    "output_dir": r"C:\Users\tlsdn\OneDrive\Documents\바탕 화면\LLM\baseline2_radiology_discharge_longformer_outputs",
    "model_name": "allenai/longformer-base-4096",
    "max_length": 4096,
    "top_k_codes": 50,
    "min_code_freq": 1,
    "batch_size": 1,              # Longformer라 1 권장
    "grad_accum_steps": 4,
    "max_epochs": 3,
    "lr_encoder": 2e-5,
    "lr_head": 1e-3,
    "warmup_ratio": 0.1,
    "threshold": 0.1,
    "grad_clip": 1.0,
    "seed": 42,
    "num_workers": 0,
    "sample_n": 2000,             # 전체 돌리려면 None
    "save_name": "best_baseline2_longformer.pt",
}

# =========================
# SEED / DEVICE
# =========================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("▶ Device:", device)

out_dir = Path(CONFIG["output_dir"])
out_dir.mkdir(parents=True, exist_ok=True)

# =========================
# HELPERS
# =========================
def prepare_text_df(df: pd.DataFrame, source_name: str):
    need_cols = {"hadm_id", "text"}
    missing = need_cols - set(df.columns)
    if missing:
        raise ValueError(f"{source_name} missing required columns: {missing}")

    df = df.dropna(subset=["hadm_id", "text"]).copy()
    df["hadm_id"] = pd.to_numeric(df["hadm_id"], errors="coerce")
    df = df.dropna(subset=["hadm_id"])
    df["hadm_id"] = df["hadm_id"].astype(int)
    df["text"] = df["text"].astype(str)

    if "charttime" in df.columns:
        df["charttime"] = pd.to_datetime(df["charttime"], errors="coerce")
    else:
        df["charttime"] = pd.NaT

    df["source"] = source_name
    return df[["hadm_id", "charttime", "text", "source"]]

# =========================
# LOAD DATA
# =========================
def load_data():
    base = Path(CONFIG["mimic_dir"])

    print("▶ Loading discharge.csv ...")
    discharge = pd.read_csv(base / "discharge.csv")
    discharge = prepare_text_df(discharge, "discharge")
    print(f"   → discharge rows: {len(discharge):,}")

    print("▶ Loading radiology.csv ...")
    radiology = pd.read_csv(base / "radiology.csv")
    radiology = prepare_text_df(radiology, "radiology")
    print(f"   → radiology rows: {len(radiology):,}")

    # radiology + discharge 결합
    docs = pd.concat([radiology, discharge], ignore_index=True)

    # 같은 hadm_id 안에서 시간순 정렬, 같은 시간이면 radiology 먼저, discharge 나중
    source_order = {"radiology": 0, "discharge": 1}
    docs["source_order"] = docs["source"].map(source_order).fillna(9).astype(int)
    docs = docs.sort_values(["hadm_id", "charttime", "source_order"], na_position="last")

    print(f"▶ Combined document rows: {len(docs):,}")

    # admission 단위 concat
    docs_concat = (
        docs.groupby("hadm_id")["text"]
        .apply(lambda xs: "\n\n".join(xs.tolist()))
        .reset_index()
    )
    print(f"   → admissions after concat: {len(docs_concat):,}")

    # labels
    print("▶ Loading diagnosis.csv ...")
    diag = pd.read_csv(base / "diagnosis.csv", usecols=["stay_id", "icd_code"])
    diag = diag.dropna(subset=["stay_id", "icd_code"]).copy()
    diag["stay_id"] = pd.to_numeric(diag["stay_id"], errors="coerce")
    diag = diag.dropna(subset=["stay_id"])
    diag["stay_id"] = diag["stay_id"].astype(int)
    diag["icd_code"] = diag["icd_code"].astype(str).str.strip()
    print(f"   → diagnosis rows: {len(diag):,}")

    print("▶ Loading edstays.csv ...")
    ed = pd.read_csv(base / "edstays.csv", usecols=["hadm_id", "stay_id"])
    ed = ed.dropna(subset=["hadm_id", "stay_id"]).copy()
    ed["hadm_id"] = pd.to_numeric(ed["hadm_id"], errors="coerce")
    ed["stay_id"] = pd.to_numeric(ed["stay_id"], errors="coerce")
    ed = ed.dropna(subset=["hadm_id", "stay_id"])
    ed["hadm_id"] = ed["hadm_id"].astype(int)
    ed["stay_id"] = ed["stay_id"].astype(int)
    print(f"   → edstays rows: {len(ed):,}")

    diag_ed = diag.merge(ed, on="stay_id", how="inner")
    print(f"   → diagnosis after bridge: {len(diag_ed):,}")

    codes_per_adm = (
        diag_ed.groupby("hadm_id")["icd_code"]
        .apply(list)
        .reset_index()
        .rename(columns={"icd_code": "labels"})
    )

    df = docs_concat.merge(codes_per_adm, on="hadm_id", how="inner")
    print(f"▶ Admissions after label merge: {len(df):,}")

    if CONFIG["sample_n"] is not None and len(df) > CONFIG["sample_n"]:
        df = df.sample(n=CONFIG["sample_n"], random_state=CONFIG["seed"]).reset_index(drop=True)
        print(f"   → Sampled admissions: {len(df):,}")

    # top-k filtering
    all_codes = [c for codes in df["labels"] for c in codes]
    freq = Counter(all_codes)

    valid_codes = {c for c, f in freq.items() if f >= CONFIG["min_code_freq"]}
    top_codes = [c for c, _ in freq.most_common(CONFIG["top_k_codes"]) if c in valid_codes]
    top_codes = sorted(top_codes)

    df["labels"] = df["labels"].apply(lambda codes: [c for c in codes if c in set(top_codes)])
    df = df[df["labels"].map(len) > 0].reset_index(drop=True)

    print(f"   → Top codes kept: {len(top_codes)}")
    print(f"   → Final admissions: {len(df):,}")

    if len(df) == 0:
        raise ValueError("No samples remain after preprocessing.")

    mlb = MultiLabelBinarizer(classes=top_codes)
    mlb.fit([top_codes])
    y = mlb.transform(df["labels"]).astype(np.float32)

    texts = df["text"].tolist()
    return texts, y, mlb, top_codes

# =========================
# DATASET
# =========================
class MIMICLongDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float32)
        return item

# =========================
# MODEL
# =========================
class LongformerMultiLabelClassifier(nn.Module):
    def __init__(self, model_name, num_classes, dropout_prob=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout_prob)
        self.classifier = nn.Linear(hidden_size, num_classes)

        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, input_ids, attention_mask, labels=None):
        global_attention_mask = torch.zeros_like(input_ids)
        global_attention_mask[:, 0] = 1

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            global_attention_mask=global_attention_mask
        )

        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        logits = self.classifier(cls_output)

        loss = None
        if labels is not None:
            loss = nn.BCEWithLogitsLoss()(logits, labels)

        return loss, logits

# =========================
# METRICS
# =========================
def precision_at_k(probs, labels, k):
    scores = []
    for i in range(len(probs)):
        top_k_idx = np.argsort(probs[i])[::-1][:k]
        true_labels = labels[i][top_k_idx]
        scores.append(true_labels.sum() / k)
    return float(np.mean(scores))

@torch.no_grad()
def evaluate(model, dataloader, threshold=0.1):
    model.eval()
    all_probs = []
    all_labels = []

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].cpu().numpy()

        _, logits = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.sigmoid(logits).cpu().numpy()

        all_probs.append(probs)
        all_labels.append(labels)

    all_probs = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)

    preds = (all_probs >= threshold).astype(int)

    micro_f1 = f1_score(all_labels, preds, average="micro", zero_division=0)
    macro_f1 = f1_score(all_labels, preds, average="macro", zero_division=0)
    p_at_5 = precision_at_k(all_probs, all_labels, 5)
    p_at_8 = precision_at_k(all_probs, all_labels, 8)

    return {
        "micro_f1": float(micro_f1),
        "macro_f1": float(macro_f1),
        "p_at_5": float(p_at_5),
        "p_at_8": float(p_at_8),
    }

# =========================
# TRAIN
# =========================
def train():
    texts, labels, mlb, code_list = load_data()
    num_classes = len(code_list)
    print(f"▶ Number of classes: {num_classes}")

    idx = np.arange(len(texts))
    idx_train, idx_temp = train_test_split(idx, test_size=0.30, random_state=CONFIG["seed"])
    idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, random_state=CONFIG["seed"])

    X_train = [texts[i] for i in idx_train]
    y_train = labels[idx_train]
    X_val = [texts[i] for i in idx_val]
    y_val = labels[idx_val]
    X_test = [texts[i] for i in idx_test]
    y_test = labels[idx_test]

    tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])

    train_ds = MIMICLongDataset(X_train, y_train, tokenizer, CONFIG["max_length"])
    val_ds = MIMICLongDataset(X_val, y_val, tokenizer, CONFIG["max_length"])
    test_ds = MIMICLongDataset(X_test, y_test, tokenizer, CONFIG["max_length"])

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=CONFIG["num_workers"])
    val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"])
    test_loader = DataLoader(test_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"])

    model = LongformerMultiLabelClassifier(CONFIG["model_name"], num_classes).to(device)

    optimizer = torch.optim.AdamW(
        [
            {"params": model.encoder.parameters(), "lr": CONFIG["lr_encoder"]},
            {"params": model.classifier.parameters(), "lr": CONFIG["lr_head"]},
        ],
        weight_decay=1e-2,
    )

    total_update_steps = (
        (len(train_loader) // CONFIG["grad_accum_steps"]) +
        int(len(train_loader) % CONFIG["grad_accum_steps"] != 0)
    ) * CONFIG["max_epochs"]

    warmup_steps = int(total_update_steps * CONFIG["warmup_ratio"])
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_update_steps)

    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

    best_val_micro = -1.0
    history = []

    print("\n▶ Training Baseline 2 (radiology + discharge + Longformer)\n")

    for epoch in range(1, CONFIG["max_epochs"] + 1):
        model.train()
        start_time = time.time()
        epoch_loss = 0.0
        optimizer.zero_grad()

        for step, batch in enumerate(train_loader, start=1):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels_batch = batch["labels"].to(device)

            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                loss, _ = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels_batch)
                loss = loss / CONFIG["grad_accum_steps"]

            scaler.scale(loss).backward()
            epoch_loss += loss.item() * CONFIG["grad_accum_steps"]

            if step % CONFIG["grad_accum_steps"] == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

            if step % 50 == 0:
                print(f"  Epoch {epoch} | Step {step}/{len(train_loader)} | Loss {epoch_loss/step:.4f}")

        val_metrics = evaluate(model, val_loader, threshold=CONFIG["threshold"])
        elapsed = time.time() - start_time

        print(
            f"\nEpoch {epoch}/{CONFIG['max_epochs']} | "
            f"Train Loss {epoch_loss/len(train_loader):.4f} | "
            f"Val Micro-F1 {val_metrics['micro_f1']:.4f} | "
            f"Val Macro-F1 {val_metrics['macro_f1']:.4f} | "
            f"P@5 {val_metrics['p_at_5']:.4f} | "
            f"P@8 {val_metrics['p_at_8']:.4f} | "
            f"Time {elapsed:.1f}s\n"
        )

        history.append({
            "epoch": epoch,
            "train_loss": epoch_loss / len(train_loader),
            **val_metrics
        })

        if val_metrics["micro_f1"] > best_val_micro:
            best_val_micro = val_metrics["micro_f1"]
            torch.save(model.state_dict(), out_dir / CONFIG["save_name"])
            print(f"  ✓ Best model saved | Val Micro-F1 {best_val_micro:.4f}\n")

    print("▶ Loading best model for test evaluation...")
    model.load_state_dict(torch.load(out_dir / CONFIG["save_name"], map_location=device))

    test_metrics = evaluate(model, test_loader, threshold=CONFIG["threshold"])

    print("\n========== BASELINE 2 TEST RESULTS ==========")
    print(f"Micro F1   : {test_metrics['micro_f1']:.4f}")
    print(f"Macro F1   : {test_metrics['macro_f1']:.4f}")
    print(f"Precision@5: {test_metrics['p_at_5']:.4f}")
    print(f"Precision@8: {test_metrics['p_at_8']:.4f}")
    print("=============================================")

    result_obj = {
        "config": CONFIG,
        "best_val_micro_f1": best_val_micro,
        "test_metrics": test_metrics,
        "history": history,
        "num_classes": num_classes,
        "num_train": len(train_ds),
        "num_val": len(val_ds),
        "num_test": len(test_ds),
        "codes": code_list,
    }

    with open(out_dir / "results_baseline2.json", "w", encoding="utf-8") as f:
        json.dump(result_obj, f, indent=2)

    print(f"▶ Saved results to: {out_dir / 'results_baseline2.json'}")

# =========================
# RUN
# =========================
train()

▶ Device: cuda
▶ Loading discharge.csv ...
   → discharge rows: 331,793
▶ Loading radiology.csv ...
   → radiology rows: 1,144,758
▶ Combined document rows: 1,476,551
   → admissions after concat: 374,285
▶ Loading diagnosis.csv ...
   → diagnosis rows: 899,050
▶ Loading edstays.csv ...
   → edstays rows: 203,016
   → diagnosis after bridge: 421,589
▶ Admissions after label merge: 177,676
   → Sampled admissions: 2,000
   → Top codes kept: 50
   → Final admissions: 1,171
▶ Number of classes: 50

▶ Training Baseline 2 (radiology + discharge + Longformer)

  Epoch 1 | Step 50/819 | Loss 0.7311
  Epoch 1 | Step 100/819 | Loss 0.6345
  Epoch 1 | Step 150/819 | Loss 0.4828
  Epoch 1 | Step 200/819 | Loss 0.3940
  Epoch 1 | Step 250/819 | Loss 0.3408
  Epoch 1 | Step 300/819 | Loss 0.3072
  Epoch 1 | Step 350/819 | Loss 0.2833
  Epoch 1 | Step 400/819 | Loss 0.2664
  Epoch 1 | Step 450/819 | Loss 0.2505
  Epoch 1 | Step 500/819 | Loss 0.2403
  Epoch 1 | Step 550/819 | Loss 0.2313
  Epoch 1 |